In [17]:
from pathlib import Path
import os
import time
import random
import torch
import pickle

import numpy as np
import pandas as pd

from training import train_single
from data_helpers import make_loader
from models_transformer import SingleOutTransformerNet
from metrics import eval_single_metrics_test

In [2]:
OUTDIR = Path("./Results")
OUTDIR.mkdir(parents=True, exist_ok=True)
MODELSDIR = Path("./trained_models")
MODELSDIR.mkdir(parents=True, exist_ok=True)

In [3]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)

Device: cpu


### Load data

In [4]:
train_data = np.load("data_processed/train_data.npz")
val_data = np.load("data_processed/val_data.npz")
test_data = np.load("data_processed/test_data.npz")

X_train = train_data["x"]
y_train = train_data["age"]

X_val = val_data["x"]
y_val = val_data["age"]

X_test = test_data["x"]
y_test = test_data["age"]

In [5]:
train_data_s = np.load("data_processed/train_data_scaled.npz")
val_data_s = np.load("data_processed/val_data_scaled.npz")
test_data_s = np.load("data_processed/test_data_scaled.npz")

X_train_s = train_data_s["x"]
y_train_s = train_data_s["age"]

X_val_s = val_data_s["x"]
y_val_s = val_data_s["age"]

X_test_s = test_data_s["x"]
y_test_s = test_data_s["age"]

In [6]:
scalers = np.load("data_processed/scalers.npz")

x_mean, x_std = scalers["x_mean"], scalers["x_std"]
y_mean, y_std = scalers["age_mean"], scalers["age_std"]

#### Slice data

In [20]:
N_LAYERS = 20
scores_df = pd.read_csv(f"Results/attention_scores/attention_scores{N_LAYERS}layers.csv")

top_k = 25
selector = scores_df["original_position"][:top_k].to_numpy()
sorted_selector = np.sort(selector)

kept_features = scores_df["feature"][selector].values.tolist()
with open(f"data_processed/dataframes/input_cols_pruned_sliced{top_k}.pkl", "wb") as f:
    pickle.dump(kept_features, f)

X_train_sliced = X_train[:, sorted_selector]
X_val_sliced = X_val[:, sorted_selector]
X_test_sliced = X_test[:, sorted_selector]

X_train_s_sliced = X_train_s[:, sorted_selector]
X_val_s_sliced = X_val_s[:, sorted_selector]
X_test_s_sliced = X_test_s[:, sorted_selector]

#### Dataloaders

In [8]:
BATCH_SIZE = 128

train_loader = make_loader(X_train_s_sliced, y_train_s, BATCH_SIZE)
val_loader = make_loader(X_val_s_sliced, y_val_s, BATCH_SIZE)
test_loader = make_loader(X_test_s_sliced, y_test_s, BATCH_SIZE)

### Models and training

In [9]:
EPOCHS = 150
# METRICS_EVERY = int(EPOCHS/8)
METRICS_EVERY = 1
LR = 1e-3
WD = 1e-5

In [10]:
IN_DIM = X_train_s_sliced.shape[1]

EMB_DIM = 64
NHEAD = 4
NUM_LAYERS = N_LAYERS
FF_DIM = 128
DROPOUT = 0.1

In [11]:
torch.manual_seed(SEED)    
model_single = SingleOutTransformerNet(IN_DIM, emb_dim=EMB_DIM, nhead=NHEAD, 
                                       num_layers=NUM_LAYERS, ff_dim=FF_DIM, 
                                       dropout=DROPOUT).to(DEVICE)    

loss = torch.nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model_single.parameters(), lr=LR, weight_decay=WD)

hist_df = train_single(model_single, loss, optimizer, train_loader, val_loader, 
                       device=DEVICE, epochs=EPOCHS, metrics_every=METRICS_EVERY)

hist_df.to_csv(f"Results/train_history/train_history_{NUM_LAYERS}layers_sliced{top_k}.csv", 
               index=False)
torch.save(model_single.state_dict(), MODELSDIR / f"TR_model_{NUM_LAYERS}layers_sliced{top_k}.pt")

ep=001 tr_loss=0.2974 | va_loss=0.2284 | tr R2=0.510 | va R2=0.486 | tr RMSE=0.700 | va RMSE=0.701 | tr MAE=0.556 | va MAE=0.554 | lr=1.00e-03 | bad_epochs=0
ep=002 tr_loss=0.2127 | va_loss=0.2009 | tr R2=0.581 | va R2=0.554 | tr RMSE=0.647 | va RMSE=0.653 | tr MAE=0.506 | va MAE=0.512 | lr=1.00e-03 | bad_epochs=0
ep=003 tr_loss=0.2100 | va_loss=0.1993 | tr R2=0.594 | va R2=0.555 | tr RMSE=0.638 | va RMSE=0.652 | tr MAE=0.495 | va MAE=0.509 | lr=1.00e-03 | bad_epochs=0
ep=004 tr_loss=0.2003 | va_loss=0.1963 | tr R2=0.593 | va R2=0.567 | tr RMSE=0.638 | va RMSE=0.644 | tr MAE=0.501 | va MAE=0.506 | lr=1.00e-03 | bad_epochs=0
ep=005 tr_loss=0.2037 | va_loss=0.2114 | tr R2=0.566 | va R2=0.521 | tr RMSE=0.659 | va RMSE=0.676 | tr MAE=0.509 | va MAE=0.523 | lr=1.00e-03 | bad_epochs=1
ep=006 tr_loss=0.1943 | va_loss=0.1970 | tr R2=0.595 | va R2=0.560 | tr RMSE=0.636 | va RMSE=0.648 | tr MAE=0.494 | va MAE=0.502 | lr=1.00e-03 | bad_epochs=2
ep=007 tr_loss=0.1919 | va_loss=0.2064 | tr R2=0.586

In [12]:
total_time = np.sum(hist_df["time"].to_numpy())
print(f"Total training time: {total_time} sec.")

Total training time: 3559.232679605484 sec.


In [13]:
m_te = eval_single_metrics_test(model_single, X_test_s_sliced, y_test, y_mean, y_std, DEVICE)

single_test_rows = []
single_test_rows.append(["age", m_te["R2"], m_te["RMSE"], m_te["MAE"]])

In [14]:
single_metrics_df = pd.DataFrame(
    single_test_rows,
    columns=["output", "R2", "RMSE", "MAE"]
)
single_metrics_path = f"Results/test_metrics/test_metrics_{NUM_LAYERS}layers_sliced{top_k}.csv"
single_metrics_df.to_csv(single_metrics_path, index=False, float_format="%.6f")

In [15]:
single_metrics_df

,output,R2,RMSE,MAE
0,age,0.619558,8.412099,6.487724
